# Combining Datasets: Concat and Append

Some of the most interesting studies of data come from combining different data sources.
These operations can involve anything from very straightforward concatenation of two different datasets, to more complicated database-style joins and merges that correctly handle any overlaps between the datasets.
``Series`` and ``DataFrame``s are built with this type of operation in mind, and Pandas includes functions and methods that make this sort of data wrangling fast and straightforward.

Here we'll take a look at simple concatenation of ``Series`` and ``DataFrame``s with the ``pd.concat`` function; later we'll dive into more sophisticated in-memory merges and joins implemented in Pandas.

We begin with the standard imports:

In [1]:
import numpy as np
import pandas as pd
df = pd.DataFrame([1,2,3,4],index = [1,3,2,4])
df.loc[1:3]

,0
1,1
3,2


In [2]:
df = pd.DataFrame([1,2,3])
df2 = df.copy()
pd.concat([df, df2], axis=1, ignore_index=False)
                  

,0,0
0,1,1
1,2,2
2,3,3


In [3]:
df1 = pd.DataFrame([[1,2,3,4], [4,3,2,1]], columns = ["A", "B", "C", "D"])
df2 = pd.DataFrame([[5,6,7,8]], columns = ["E", "B", "C", "D"])
pd.concat([df1, df2], axis = 0, join= "inner")
 

,B,C,D
0,2,3,4
1,3,2,1
0,6,7,8


In [4]:
pd.concat([df1, df2],axis=1)

,A,B,C,D,E,B,C,D
0,1,2,3,4,5.0,6.0,7.0,8.0
1,4,3,2,1,NaN,NaN,NaN,NaN


## Simple Concatenation with ``pd.concat``

Pandas has a function, ``pd.concat()``, which has a similar syntax to ``np.concatenate`` but contains a number of options that we'll discuss momentarily:

```python
# Signature in Pandas v0.18
pd.concat(objs, axis=0, join='outer', join_axes=None, ignore_index=False,
          keys=None, levels=None, names=None, verify_integrity=False,
          copy=True)
```

``pd.concat()`` can be used for a simple concatenation of ``Series`` or ``DataFrame`` objects, just as ``np.concatenate()`` can be used for simple concatenations of arrays:

In [5]:
s1 = pd.DataFrame([11,22,33])
s2 = s1.copy()*5
s3 = pd.concat([s1,s2], axis= 1, ignore_index= True, verify_integrity= True)
s3

,0,1
0,11,55
1,22,110
2,33,165


In [6]:
s1 = pd.Series([11,22,33], index=list("ABC"))
s2 = s1.copy()
s3 = pd.concat([s1,s2]).reset_index(drop=False)
s3.drop_duplicates()

,index,0
0,A,11
1,B,22
2,C,33


In [7]:
s3

,index,0
0,A,11
1,B,22
2,C,33
3,A,11
4,B,22
5,C,33


In [8]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1,2,3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4,5,6])
pd.concat([ser1, ser2], axis = 0)


1    A
2    B
3    C
4    D
5    E
6    F
dtype: str

In [9]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1,2,3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4,5,6])
pd.concat([ser1, ser2], axis = 0)

1    A
2    B
3    C
4    D
5    E
6    F
dtype: str

It also works to concatenate higher-dimensional objects, such as ``DataFrame``s:

In [10]:
print(df)

df_otro = df.copy()
df_otro.loc[1,"palabras"] = "A"
df_otro

   0
0  1
1  2
2  3


,0,palabras
0,1,NaN
1,2,A
2,3,NaN


In [11]:
print(df)
print(df_otro)


pd.concat([df, df_otro], join= "outer", axis="columns")

   0
0  1
1  2
2  3
   0 palabras
0  1      NaN
1  2        A
2  3      NaN


,0,0,palabras
0,1,1,NaN
1,2,2,A
2,3,3,NaN


By default, the concatenation takes place row-wise within the ``DataFrame`` (i.e., ``axis=0``).
Like ``np.concatenate``, ``pd.concat`` allows specification of an axis along which concatenation will take place.
Consider the following example:

In [12]:
df3 = pd.DataFrame(list('AB'), [0,2])
df4 = pd.DataFrame(list('CD'), [0,1])

pd.concat([df3, df4], axis= 0, join = "outer", ignore_index= True)

,0
0,A
1,B
2,C
3,D


We could have equivalently specified ``axis=1``; here we've used the more intuitive ``axis='col'``. 

### Duplicate indices

One important difference between ``np.concatenate`` and ``pd.concat`` is that Pandas concatenation *preserves indices*, even if the result will have duplicate indices!
Consider this simple example:

In [13]:
x = pd.Series([11,22,33], index=list("ABC"))
y = s1.copy()
s3 = pd.concat([s1,s2]).reset_index(drop=False)
s3.drop_duplicates()

,index,0
0,A,11
1,B,22
2,C,33


Notice the repeated indices in the result.
While this is valid within ``DataFrame``s, the outcome is often undesirable.
``pd.concat()`` gives us a few ways to handle it.

#### Catching the repeats as an error

If you'd like to simply verify that the indices in the result of ``pd.concat()`` do not overlap, you can specify the ``verify_integrity`` flag.
With this set to True, the concatenation will raise an exception if there are duplicate indices.
Here is an example, where for clarity we'll catch and print the error message:

In [14]:
try:
    pd.concat([x,y], verify_integrity=True)
except ValueError:
        pd.concat([x,y], ignore_index=True)

#### Ignoring the index

Sometimes the index itself does not matter, and you would prefer it to simply be ignored.
This option can be specified using the ``ignore_index`` flag.
With this set to true, the concatenation will create a new integer index for the resulting ``Series``:

In [15]:
pd.concat([x, y], ignore_index=True)

0    11
1    22
2    33
3    11
4    22
5    33
dtype: int64

#### Adding MultiIndex keys

Another option is to use the ``keys`` option to specify a label for the data sources; the result will be a hierarchically indexed series containing the data:

In [ ]:
a = pd.DataFrame([1,2,3,4,5])
b = pd.DataFrame([1,2,3,4,5])


x = pd.concat([a, b], keys=[4, 5])

x[(4,1), 0]     # Aquí accedemos con una tumpla al multiíndice

0
4 0  1
  1  2
  2  3
  3  4
  4  5
5 0  1
  1  2
  2  3
  3  4
  4  5

The result is a multiply indexed ``DataFrame``, and we can use the tools discussed in [Hierarchical Indexing](03.05-Hierarchical-Indexing.ipynb) to transform this data into the representation we're interested in.

### Concatenation with joins

In the simple examples we just looked at, we were mainly concatenating ``DataFrame``s with shared column names.
In practice, data from different sources might have different sets of column names, and ``pd.concat`` offers several options in this case.
Consider the concatenation of the following two ``DataFrame``s, which have some (but not all!) columns in common:

In [28]:
df5 = pd.DataFrame(list('ABC'), [1, 2, 6])
df6 = pd.DataFrame(list('BCD'), [3, 4, 5])
display(pd.concat([df5, df6]))

,0
1,A
2,B
6,C
3,B
4,C
5,D


By default, the entries for which no data is available are filled with NA values.
To change this, we can specify one of several options for the ``join`` and ``join_axes`` parameters of the concatenate function.
By default, the join is a union of the input columns (``join='outer'``), but we can change this to an intersection of the columns using ``join='inner'``:

In [30]:
pd.concat([df5, df6], join='inner', sort = True)  
# El sort no hace efecto ya que esta actando sobre las columnas, 
# el contrario del que actua el concat

,0
1,A
2,B
6,C
3,B
4,C
5,D


The combination of options of the ``pd.concat`` function allows a wide range of possible behaviors when joining two datasets; keep these in mind as you use these tools for your own data.